In [1]:
import json
import requests

In [2]:
ANKI_URL = "http://127.0.0.1:8765"

In [3]:
def request(action, **params):
    """构造请求体"""
    return {"action": action, "params": params, "version": 6}

In [4]:
def invoke(action, **params):
    """发送请求并返回结果"""
    payload = request(action, **params)
    try:
        response = requests.post(ANKI_URL, json=payload, timeout=5)
    except requests.exceptions.RequestException as e:
        raise SystemExit(f"❌ 无法连接 AnkiConnect: {e}")

    if response.status_code != 200:
        raise Exception(f"HTTP错误: {response.status_code}")

    data = response.json()

    # 按照 Anki-Connect 格式校验返回值
    if not isinstance(data, dict):
        raise Exception("响应格式错误：不是 JSON 对象")
    if "error" not in data or "result" not in data:
        raise Exception("响应缺少必要字段 'error' 或 'result'")
    if data["error"] is not None:
        raise Exception(f"AnkiConnect 返回错误: {data['error']}")

    return data["result"]

In [6]:
invoke("createDeck", deck="test1")

1759707976865

In [7]:
decks = invoke("deckNames")
print("当前牌组列表:", decks)

当前牌组列表: ['1st example', '1st example::reservse', '4000 Essential English Words', '4000 Essential English Words::1.Book', '4000 Essential English Words::2.Book', '4000 Essential English Words::3.Book', '4000 Essential English Words::4.Book', '4000 Essential English Words::5.Book', '4000 Essential English Words::6.Book', '4000 Essential English Words::Extra', 'Beginning Database Design', 'Everyday English', 'oule', 'Saladict', 'test1', '系统默认']


In [9]:
invoke("deleteDecks", decks=["test1"], cardsToo="true")

In [11]:
def ensure_deck(deck_name: str):
    decks = set(invoke("deckNames"))
    if deck_name not in decks:
        invoke("createDeck", deck=deck_name)

In [12]:
def get_model_fields(model_name: str) -> list[str]:
    return invoke("modelFieldNames", modelName=model_name)

In [21]:
def build_fields(model_fields: list[str], provided: dict) -> dict:
    """
    根据实际模型字段名做映射。
    你可以只提供通用键（term/gloss/example/phonetic/sound/image），
    其余字段置空；若你明确知道字段名，也可直接提供同名键覆盖。
    """
    fields = {f: "" for f in model_fields}

    # ---- 先把用户显式提供的同名字段直接写入（强覆盖）----
    for k, v in provided.items():
        if k in fields:
            fields[k] = v

    # ---- 通用键尝试映射到常见字段名（仅当目标字段仍为空）----
    # 1) 词条/正面文本
    if "term" in provided:
        for cand in ["Word", "Expression", "Term", "Headword", "Front", "单词", "词条"]:
            if cand in fields and not fields[cand]:
                fields[cand] = provided["term"]
                break

    # 2) 音标
    if "phonetic" in provided:
        for cand in ["Phonetic", "IPA", "音标"]:
            if cand in fields and not fields[cand]:
                fields[cand] = provided["phonetic"]
                break

    # 3) 释义/中文释义
    if "gloss" in provided:
        for cand in ["Definition", "Meaning", "Gloss", "释义", "中文释义", "词典释义", "Back"]:
            if cand in fields and not fields[cand]:
                fields[cand] = provided["gloss"]
                break

    # 4) 例句
    if "example" in provided:
        for cand in ["Sentence", "Example", "Examples", "例句"]:
            if cand in fields and not fields[cand]:
                fields[cand] = provided["example"]
                break

    # 5) 近义/搭配（可选）
    if "collocation" in provided:
        for cand in ["Collocation", "Phrases", "搭配"]:
            if cand in fields and not fields[cand]:
                fields[cand] = provided["collocation"]
                break

    return fields

In [20]:
def add_note(
    deck_name="oule",
    model_name="FastWQ",
    provided_fields: dict | None = None,
    tags: list[str] | None = None,
    audio: dict | None = None,
    picture: dict | None = None,
) -> int:
    """
    :param provided_fields: 既可用“模型同名字段”，也可用通用键：
        term / gloss / example / phonetic / collocation
        （例如 provided_fields = {"term":"aurora","gloss":"极光","example":"We saw ..."}）
    :param audio: 可选，如 {"url": "...mp3", "filename": "aurora.mp3", "fields": ["Sound"]}
    :param picture: 可选，如 {"url": "...jpg", "filename": "aurora.jpg", "fields": ["Image"]}
    :return: 新增笔记的 noteId
    """
    provided_fields = provided_fields or {}
    tags = tags or ["FastWQ", "auto"]

    # 1) 确保牌组存在
    ensure_deck(deck_name)

    # 2) 获取模型字段并构造字段映射
    model_fields = get_model_fields(model_name)  # 若模型不存在，这里会抛出异常
    fields = build_fields(model_fields, provided_fields)

    # 3) 组装 note
    note = {
        "deckName": deck_name,
        "modelName": model_name,
        "fields": fields,
        "tags": tags,
        "options": {
            "allowDuplicate": False,
            "duplicateScope": "deck",  # 在本牌组内查重
        },
    }

    # 附件（可选）
    if audio:
        note["audio"] = [{
            "url": audio["url"],
            "filename": audio["filename"],
            "fields": audio.get("fields", []),
        }]
    if picture:
        note["picture"] = [{
            "url": picture["url"],
            "filename": picture["filename"],
            "fields": picture.get("fields", []),
        }]

    # 4) 调用 addNote
    note_id = invoke("addNote", note=note)
    return note_id

In [15]:
provided = {
            # 通用键（自动匹配模型字段）
            "term": "aurora",
            "phonetic": "/əˈrɔːrə/",
            "gloss": "a natural light display in the Earth's sky; 极光",
            "example": "We saw the aurora dancing over the Arctic.",
            # 如果你知道 FastWQ 的确切字段名，也可以直接用同名键覆盖：
            # "Word": "aurora",
            # "Definition": "a natural light display ...",
            # "Example": "We saw ...",
            # "Phonetic": "/əˈrɔːrə/",
        }

In [ ]:
note_id = add_note(
            deck_name="oule",
            model_name="FastWQ",
            provided_fields=provided,
            tags=["oulu", "FastWQ", "auto"],
            # audio=audio,
            # picture=picture,
        )

In [16]:
print("FastWQ model fields:",get_model_fields(model_name="FastWQ"))

FastWQ model fields: ['单词', '词典释义', '发音', '音标', '例句']


In [19]:
FieldDescriptions = invoke("modelFieldDescriptions", modelName="FastWQ")
print("FastWQ模型的域说明的列表:", FieldDescriptions)

FastWQ模型的域说明的列表: ['', '', '', '', '']
